# Rainfall Classification: Pipelines Comparison

This notebook reproduces and cleans the original rain pipeline experiments:
- Baseline GBM
- PCA + GBM
- LDA + GBM
- Hyperparameter tuning for LDA + GBM
- LdaBoost model evaluation (from the local `LdaBoost` package)

All code and text are in English, with reproducible seeds and structured sections.


In [ ]:
# Reproducible setup and imports
import os
import random
import logging
import re


import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier

import optuna

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Global seed
RANDOM_SEED: int = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Import LdaBoost from local package via relative path
import sys
REL_PROJECT_ROOT = "../../"
if REL_PROJECT_ROOT not in sys.path:
    sys.path.insert(0, REL_PROJECT_ROOT)
from LdaBoost import LdaBoost


In [8]:
N_ESTIMATOR = 100
LEARNING_RATE = 0.1
MAX_DEPTH = 3

## Dataset and preprocessing


In [9]:
def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [re.sub(r"[^A-Za-z0-9_]+", "_", c) for c in df.columns]
    return df

# Load and clean
train_df = pd.read_csv("Data/train.csv", index_col="id")
train_df = clean_column_names(train_df)

X: pd.DataFrame = train_df.drop(columns="rainfall")
y: pd.Series = train_df["rainfall"]

le = LabelEncoder()
y_enc: np.ndarray = le.fit_transform(y)

features: list[str] = list(X.columns)
cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)



## Baseline: GBM


In [10]:
scaler_pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),
])

base_preprocess = ColumnTransformer(
    remainder="passthrough",
    transformers=[("scale", scaler_pipe, features)],
)

base_boost = Pipeline(steps=[
    ("preprocess", base_preprocess),
    (
        "gb",
        GradientBoostingClassifier(
            n_estimators=N_ESTIMATOR,
            learning_rate=LEARNING_RATE,
            max_depth=MAX_DEPTH,
            random_state=RANDOM_SEED,
        ),
    ),
])

acc_scores = cross_val_score(base_boost, X, y_enc, cv=cv10, scoring="accuracy")
f1_scores = cross_val_score(base_boost, X, y_enc, cv=cv10, scoring="f1_macro")

print(f"Accuracy (10-fold): mean={acc_scores.mean():.4f} std={acc_scores.std():.4f}")
print(f"F1 macro (10-fold): mean={f1_scores.mean():.4f} std={f1_scores.std():.4f}")

baseline_results: dict[str, np.ndarray] = {"accuracy": acc_scores, "f1_macro": f1_scores}



Accuracy (10-fold): mean=0.8607 std=0.0254
F1 macro (10-fold): mean=0.8042 std=0.0368


## PCA + GBM


In [11]:
pca_pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=max(1, X.shape[1] // 2), random_state=RANDOM_SEED)),
])

pca_preprocess = ColumnTransformer(
    remainder="drop",
    transformers=[("pca", pca_pipe, features)],
)

pca_boost = Pipeline(steps=[
    ("preprocess", pca_preprocess),
    ("gb", GradientBoostingClassifier(
            n_estimators=N_ESTIMATOR,
            learning_rate=LEARNING_RATE,
            max_depth=MAX_DEPTH,
            random_state=RANDOM_SEED)),
])

acc_scores = cross_val_score(pca_boost, X, y_enc, cv=cv10, scoring="accuracy")
f1_scores = cross_val_score(pca_boost, X, y_enc, cv=cv10, scoring="f1_macro")

print(f"Accuracy (10-fold): mean={acc_scores.mean():.4f} std={acc_scores.std():.4f}")
print(f"F1 macro (10-fold): mean={f1_scores.mean():.4f} std={f1_scores.std():.4f}")

pca_results: dict[str, np.ndarray] = {"accuracy": acc_scores, "f1_macro": f1_scores}



Accuracy (10-fold): mean=0.8603 std=0.0187
F1 macro (10-fold): mean=0.8041 std=0.0282


## LDA + GBM


In [12]:
class LDATransformer:
    def __init__(self, n_components: int | None = None):
        self.n_components = n_components
        self.lda: LinearDiscriminantAnalysis | None = None
    def fit(self, X, y):
        self.lda = LinearDiscriminantAnalysis(n_components=self.n_components)
        self.lda.fit(X, y)
        return self
    def transform(self, X):
        assert self.lda is not None
        return self.lda.transform(X)
    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)

lda_pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("lda", LDATransformer(n_components=None)),
])

lda_preprocess = ColumnTransformer(
    remainder="drop",
    transformers=[("lda", lda_pipe, list(range(X.shape[1])))],
)

lda_boost = Pipeline(steps=[
    ("preprocess", lda_preprocess),
    ("gb", GradientBoostingClassifier(
            n_estimators=N_ESTIMATOR,
            learning_rate=LEARNING_RATE,
            max_depth=MAX_DEPTH,
            random_state=RANDOM_SEED)),
])

acc_scores = cross_val_score(lda_boost, X, y_enc, cv=cv10, scoring="accuracy")
f1_scores = cross_val_score(lda_boost, X, y_enc, cv=cv10, scoring="f1_macro")

print(f"Accuracy (10-fold): mean={acc_scores.mean():.4f} std={acc_scores.std():.4f}")
print(f"F1 macro (10-fold): mean={f1_scores.mean():.4f} std={f1_scores.std():.4f}")

lda_results: dict[str, np.ndarray] = {"accuracy": acc_scores, "f1_macro": f1_scores}



Accuracy (10-fold): mean=0.8557 std=0.0237
F1 macro (10-fold): mean=0.7985 std=0.0346


## LdaBoost evaluation


In [13]:
X_np: np.ndarray = X.to_numpy()

acc_scores = []
f1_scores = []
for tr_idx, te_idx in cv10.split(X_np, y_enc):
    X_tr, X_te = X_np[tr_idx], X_np[te_idx]
    y_tr, y_te = y_enc[tr_idx], y_enc[te_idx]

    model = LdaBoost(
            n_estimators=N_ESTIMATOR,
            learning_rate=LEARNING_RATE,
            max_depth=MAX_DEPTH,
            random_state=RANDOM_SEED)
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    acc_scores.append(accuracy_score(y_te, y_pred))
    f1_scores.append(f1_score(y_te, y_pred, average="macro"))

print(f"LdaBoost — Accuracy (10-fold): mean={np.mean(acc_scores):.4f} std={np.std(acc_scores):.4f}")
print(f"LdaBoost — F1 macro (10-fold): mean={np.mean(f1_scores):.4f} std={np.std(f1_scores):.4f}")



LdaBoost — Accuracy (10-fold): mean=0.8635 std=0.0251
LdaBoost — F1 macro (10-fold): mean=0.8100 std=0.0360
